# 03 · MNIST final training

Notebook này đọc thư mục dataset, train mọi model trong `MODEL_CONFIGS`, lưu file weight và đánh giá trên validation/test.

In [ ]:
from pathlib import Path
import sys
import json
from time import perf_counter
import numpy as np
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / 'gk' / 'web' / 'backend').exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.preprocessing import StandardScaler
from gk.web.backend.digits_config import (
    DATASET_DIR, DATASET_PATH, MODEL_ARTIFACT_PATH, MODEL_CONFIG_PATH, RAW_DATASET_PATH,
    PIXEL_COUNT, PIXEL_SIZE, SPLIT_PATH, WEIGHTS_PATH,
)
from gk.web.backend.digits_models import build_logistic, build_mlp, export_layers, model_metadata
from gk.web.backend.digits_augmentation import build_augmented_variants, repeat_labels
from gk.web.backend.model_config import (
    AUGMENT_FACTOR, AUGMENT_ROTATE_DEGREES, AUGMENT_SCALE_RANGE, AUGMENT_SHARPEN,
    AUGMENT_SHIFT_PIXELS, AUGMENT_STROKE_VARIANTS, AUGMENT_TRAINING,
    architecture_sizes, hidden_activation, hidden_layer_sizes, layer_activations,
)

CONFIG_PATH = MODEL_CONFIG_PATH
OUTPUT_PATH = WEIGHTS_PATH
TRAIN_LOG_PATH = DATASET_DIR / 'training_log.json'
WEB_OUTPUT_PATH = MODEL_ARTIFACT_PATH
print('Configured input:', f'{PIXEL_SIZE}x{PIXEL_SIZE} ({PIXEL_COUNT} features)')
data = np.load(DATASET_PATH)
raw_data = np.load(RAW_DATASET_PATH)
if not CONFIG_PATH.exists():
    raise FileNotFoundError(f'Run mnist_models.ipynb first to create {CONFIG_PATH}')
config = json.loads(CONFIG_PATH.read_text(encoding='utf-8'))
X_train_images = data['X_train_pixels'].astype(float)
y_train = data['y_train'].astype(int)
X_test_images = data['X_test_pixels'].astype(float)
y_test = data['y_test'].astype(int)
X_train_pixels = X_train_images.reshape(len(X_train_images), -1)
X_test_pixels = X_test_images.reshape(len(X_test_images), -1)
split = json.loads(SPLIT_PATH.read_text(encoding='utf-8'))
fit_indices = np.asarray(split['fit_indices'], dtype=int)
validation_indices = np.asarray(split['validation_indices'], dtype=int)
raw_train_images = raw_data['X_train_images'].astype(float)
original_fit_pixels = X_train_pixels[fit_indices]
original_fit_labels = y_train[fit_indices]
if AUGMENT_TRAINING:
    augmented_pixels = build_augmented_variants(
        raw_train_images[fit_indices],
        factor=AUGMENT_FACTOR,
        shift_pixels=AUGMENT_SHIFT_PIXELS,
        stroke_variants=AUGMENT_STROKE_VARIANTS,
        scale_range=AUGMENT_SCALE_RANGE,
        rotate_degrees=AUGMENT_ROTATE_DEGREES,
        sharpen=AUGMENT_SHARPEN,
        seed=config['random_seed'],
    )
    fit_pixels = np.vstack((original_fit_pixels, augmented_pixels))
    y_fit = repeat_labels(original_fit_labels, AUGMENT_FACTOR)
else:
    augmented_pixels = np.empty((0, PIXEL_COUNT), dtype=np.float32)
    fit_pixels = original_fit_pixels
    y_fit = original_fit_labels
scaler = StandardScaler().fit(fit_pixels)
mean = scaler.mean_
std = scaler.scale_
config['scaler'] = {'mean': mean.tolist(), 'std': std.tolist()}
CONFIG_PATH.write_text(json.dumps(config, indent=2), encoding='utf-8')
X_fit = scaler.transform(fit_pixels)
X_validation = scaler.transform(X_train_pixels[validation_indices])
X_test = scaler.transform(X_test_pixels)
y_validation = y_train[validation_indices]
print('Loaded dataset and model config')
print(f'augmentation: {len(original_fit_pixels)} originals + {len(augmented_pixels)} variants = {len(X_fit)} train samples')


## Train bốn model và ghi log

Các config ở đây đến từ `mnist_model_config.json`, vì vậy có thể tối ưu notebook model mà không sửa notebook train. Log bên dưới lấy trực tiếp từ object đã fit.

In [ ]:
results = {}
for model_id, model in trained.items():
    results[model_id] = {
        'validation_accuracy': accuracy_score(y_validation, model.predict(X_validation)),
        'test_accuracy': accuracy_score(y_test, model.predict(X_test)),
    }
    record = next(item for item in training_log if item['model_id'] == model_id)
    record.update(results[model_id])
    print(f'{model_id:24} validation={results[model_id]["validation_accuracy"]:.2%} test={results[model_id]["test_accuracy"]:.2%} elapsed={record["elapsed_seconds"]:.2f}s')
TRAIN_LOG_PATH.write_text(json.dumps({
    'dataset': str(DATASET_PATH), 'train_samples': len(X_fit),
    'original_train_samples': len(original_fit_pixels),
    'augmented_samples': len(augmented_pixels),
    'augmentation': {
        'enabled': AUGMENT_TRAINING, 'factor': AUGMENT_FACTOR,
        'shift_pixels': AUGMENT_SHIFT_PIXELS,
        'stroke_variants': AUGMENT_STROKE_VARIANTS,
        'scale_range': list(AUGMENT_SCALE_RANGE),
    },
    'validation_samples': len(X_validation), 'test_samples': len(X_test),
    'models': training_log,
}, indent=2), encoding='utf-8')
print('saved training log:', TRAIN_LOG_PATH)

## Confusion matrix và loss curve

In [ ]:
fig, axes = plt.subplots(1, len(trained), figsize=(3.5 * len(trained), 3))
if len(trained) == 1:
    axes = [axes]
for axis, (model_id, model) in zip(axes, trained.items()):
    matrix = confusion_matrix(y_test, model.predict(X_test))
    axis.imshow(matrix, cmap='Blues')
    axis.set_title(model_id)
    axis.set_xlabel('predicted')
    axis.set_ylabel('true')
plt.tight_layout()
plt.show()

In [ ]:
sample_index = 7
plt.figure(figsize=(2, 2))
plt.imshow(X_test_pixels[sample_index].reshape(PIXEL_SIZE, PIXEL_SIZE), cmap='gray_r', vmin=0, vmax=1)
plt.title(f'true label: {y_test[sample_index]}')
plt.axis('off')
plt.show()
for model_id, model in trained.items():
    probabilities = model.predict_proba(X_test[sample_index:sample_index + 1])[0]
    print(model_id, '→', int(np.argmax(probabilities)), f'{probabilities.max():.2%}')

## Export artifact cho web

In [ ]:
baseline_predictions = trained['logistic'].predict(X_test)
strong_ann_id = [model_id for model_id, cfg in model_configs.items() if cfg.get('kind') == 'ann'][-1]
tuned_predictions = trained[strong_ann_id].predict(X_test)
demo_sample_index = 0
for index, baseline_prediction, tuned_prediction, target in zip(
    range(len(y_test)), baseline_predictions, tuned_predictions, y_test
):
    if baseline_prediction != target and tuned_prediction == target:
        demo_sample_index = int(index)
        break

def make_metadata(model_id, name, kind, model, architecture, activations, hidden_act, source=None, source_url=None):
    return model_metadata(
        model_id, name, kind, model, architecture, activations,
        results[model_id]['test_accuracy'],
        results[model_id]['validation_accuracy'],
        hidden_act, source, source_url,
    )

def ann_display_name(model_id, model_config):
    hidden = list(hidden_layer_sizes(model_config))
    activation = hidden_activation(model_config)
    if len(hidden) == 1:
        return f'MLP · {hidden[0]} · {activation}'
    return f"MLP · {'×'.join(str(size) for size in hidden)} · {activation}"

models = []
for model_id, model_config in model_configs.items():
    if model_config.get('kind') == 'linear':
        models.append(make_metadata(
            model_id, 'Logistic Regression', 'linear', trained[model_id],
            architecture_sizes(model_config), [], None,
            'scikit-learn LogisticRegression',
            'https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html',
        ))
        continue
    models.append(make_metadata(
        model_id, ann_display_name(model_id, model_config), 'ann', trained[model_id],
        architecture_sizes(model_config), layer_activations(model_config), hidden_activation(model_config),
    ))

artifact = {
    'version': 4,
    'random_seed': config['random_seed'],
    'pixel_size': PIXEL_SIZE,
    'dataset': {
        'name': 'MNIST handwritten digits',
        'description': f'Real handwritten digit images from MNIST, normalized to {PIXEL_SIZE}x{PIXEL_SIZE} grayscale pixels for this demo.',
        'sample_count': 70000, 'training_sample_count': 60000, 'test_sample_count': 10000,
        'input_shape': [PIXEL_SIZE, PIXEL_SIZE], 'feature_count': PIXEL_COUNT, 'class_count': 10,
        'pixel_min': 0, 'pixel_max': 1, 'original_input_shape': [28, 28],
        'source_url': 'https://yann.lecun.com/exdb/mnist/',
    },
    'preprocessing': {
        'name': f'CropSquareResize{PIXEL_SIZE} + StandardScaler',
        'feature_transform': f'denoise at 50% of max ink, keep largest component, crop foreground, resize to square, area-average resize to {PIXEL_SIZE}x{PIXEL_SIZE}, scale intensity to 0..1',
        'mean': mean.tolist(), 'std': std.tolist(),
    },
    'fit_indices': config['fit_indices'], 'validation_indices': config['validation_indices'],
    'test_indices': config['test_indices'],
    'test_samples': X_test_pixels.astype(float).tolist(),
    'test_labels': y_test.astype(int).tolist(),
    'demo_sample_index': demo_sample_index,
    'augmentation': {
        'enabled': AUGMENT_TRAINING,
        'factor': AUGMENT_FACTOR,
        'source_count': int(len(original_fit_pixels)),
        'generated_count': int(len(X_fit)),
        'variant_count': int(len(augmented_pixels)),
        'shift_pixels': AUGMENT_SHIFT_PIXELS,
        'stroke_variants': AUGMENT_STROKE_VARIANTS,
        'scale_range': list(AUGMENT_SCALE_RANGE),
        'rotate_degrees': AUGMENT_ROTATE_DEGREES,
        'sharpen': AUGMENT_SHARPEN,
        'applied_to': 'fit split only; validation and test remain unchanged',
    },
    'baseline_tuning': {
        'rounds': config['search']['logistic_rounds'],
        'C_candidates': [row['C'] for row in config['search']['logistic_candidates']],
        'selected_C': model_configs['logistic']['C'],
        'validation_accuracy': config['search']['selected_logistic_validation_accuracy'],
    },
    'models': models,
    'model_configs': {
        model_id: {
            'kind': model_config.get('kind'),
            'architecture': architecture_sizes(model_config),
            'activations': layer_activations(model_config),
        }
        for model_id, model_config in model_configs.items()
    },
    'strong_ann_id': strong_ann_id,
}
serialized_artifact = json.dumps(artifact, indent=2)
OUTPUT_PATH.write_text(serialized_artifact, encoding='utf-8')
WEB_OUTPUT_PATH.write_text(serialized_artifact, encoding='utf-8')
print(f'Wrote weights: {OUTPUT_PATH}')
print(f'Updated web artifact: {WEB_OUTPUT_PATH}')
